# Fine-tuning avec LoRA (Low-Rank Adaptation)

## Ce que tu vas apprendre
1. Appliquer LoRA à un modèle de langage pré-entraîné.
2. Fine-tuner un modèle adapté avec LoRA via la librairie Hugging Face **PEFT**.
3. Sauvegarder et recharger un modèle LoRA fine-tuné.
4. Faire de l'inférence avec ce modèle.

## Ce que tu vas créer
Un modèle de langage fine-tuné pour générer du texte dans le style de citations (dataset `Abirate/english_quotes`), en utilisant LoRA.

**Remarque avant de commencer (et je te le dis franchement) :** cet exercice utilise volontairement des valeurs extrêmes (`r=1`, `learning_rate=3e-2`, seulement quelques exemples d'entraînement) pour que ça tourne vite sur CPU. Ce n'est PAS une configuration réaliste pour obtenir un bon modèle — c'est une démo pédagogique. Ne réutilise pas ces hyperparamètres tels quels sur un vrai projet, tu obtiendrais un modèle sous-entraîné qui ne fait quasiment que du bruit.

## Étape 0 : Installation des librairies nécessaires

In [ ]:
%pip install peft==0.4.0 transformers datasets accelerate -q

In [ ]:
import os

# On crée un dossier cache local pour stocker les modèles et les sorties d'entraînement
os.makedirs("../cache/working", exist_ok=True)
print("Dossier cache prêt.")

## Étape 1 : Charger le modèle pré-entraîné et son tokenizer

On utilise `bigscience/bloomz-560m`, un petit modèle causal (560 millions de paramètres) qui tient sur CPU.

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "bigscience/bloomz-560m"

# Le tokenizer transforme le texte en identifiants numériques compréhensibles par le modèle
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Le modèle "foundation" est le modèle de base, avant tout fine-tuning
foundation_model = AutoModelForCausalLM.from_pretrained(model_name)

print(foundation_model.config)

## Étape 2 : Charger et préparer le dataset

Le dataset `Abirate/english_quotes` contient des citations en anglais. On va :
1. Le charger.
2. Le tokenizer (convertir chaque citation en tokens).
3. En garder seulement 10% pour aller vite (c'est un exercice de démo, pas un vrai entraînement).

In [ ]:
from datasets import load_dataset

data = load_dataset("Abirate/english_quotes")

# On tokenize chaque exemple du split "train" (batched=True = traitement par lots, plus rapide)
data = data.map(lambda samples: tokenizer(samples["quote"]), batched=True)

# Le tokenizer de bloomz n'a pas de pad_token par défaut : on lui en donne un
# (nécessaire plus tard pour le DataCollator)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# On mélange puis on garde 10% du split train
full_train = data["train"].shuffle(seed=42)
sample_size = int(0.10 * len(full_train))
train_sample = full_train.select(range(sample_size))

print(f"Taille totale du train : {len(full_train)}")
print(f"Taille de l'échantillon (10%) : {len(train_sample)}")

# Petit aperçu des 5 premiers exemples
display(train_sample.select(range(5)))

## Étape 3 : Configurer LoRA

LoRA n'entraîne pas tous les poids du modèle. Il ajoute des petites matrices de rang faible ("low-rank") à certaines couches, et seules ces matrices sont entraînées. Ça réduit énormément le nombre de paramètres à entraîner.

- `r` : le rang des matrices LoRA. Plus il est petit, moins on entraîne de paramètres (mais moins le modèle peut apprendre). On met `r=1` ici pour que ça tourne vite sur CPU — en pratique, on utilise souvent `r=8` ou `r=16`.
- `lora_alpha` : facteur d'échelle appliqué aux poids LoRA. Souvent égal à `r` ou à un multiple de `r`. Ici on suit la consigne : `1`.
- `target_modules` : les couches du modèle sur lesquelles on applique LoRA. Pour l'architecture Bloom, la couche d'attention combinée s'appelle `query_key_value`.
- `lora_dropout` : dropout appliqué aux couches LoRA pour limiter le sur-apprentissage.
- `bias="none"` : on n'entraîne pas les biais.
- `task_type="CAUSAL_LM"` : on précise qu'on fait de la génération de texte (modèle causal, qui prédit le mot suivant).

In [ ]:
import peft
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=1,
    lora_alpha=1,  # facteur d'échelle appliqué au poids LoRA, souvent égal à r
    target_modules=["query_key_value"],  # couche d'attention de Bloom
    lora_dropout=0.05,
    bias="none",  # on ne touche pas aux biais
    task_type="CAUSAL_LM"
)

## Étape 4 : Appliquer LoRA au modèle de base

In [ ]:
# On "greffe" les couches LoRA sur le modèle de base
peft_model = get_peft_model(foundation_model, lora_config)

# Affiche combien de paramètres sont réellement entraînables (ça devrait être une toute petite fraction du total)
print(peft_model.print_trainable_parameters())

## Étape 5 : Configurer l'entraînement

In [ ]:
import transformers
from transformers import TrainingArguments, Trainer

output_directory = os.path.join("../cache/working", "peft_lab_outputs")

training_args = TrainingArguments(
    report_to="none",
    output_dir=output_directory,
    auto_find_batch_size=True,  # laisse Transformers trouver automatiquement une taille de batch qui rentre en mémoire
    learning_rate=3e-2,  # taux d'apprentissage volontairement élevé (plus haut que pour un fine-tuning complet)
    num_train_epochs=5,  # peu d'epochs car le dataset d'entraînement est minuscule (démo)
    use_cpu=True
)

trainer = Trainer(
    model=peft_model,
    args=training_args,
    train_dataset=train_sample,
    data_collator=transformers.DataCollatorForLanguageModeling(tokenizer, mlm=False)
    # mlm=False -> on fait du "causal language modeling" (prédire le mot suivant),
    # pas du "masked language modeling" (style BERT)
)

## Étape 6 : Lancer l'entraînement

⚠️ Sur CPU, même avec un tout petit dataset, ça peut prendre plusieurs minutes.

In [ ]:
trainer.train()

## Étape 7 : Sauvegarder le modèle LoRA fine-tuné

On ne sauvegarde que les poids LoRA (les petites matrices), pas le modèle complet. C'est un des gros avantages de LoRA : les fichiers sauvegardés sont légers.

In [ ]:
import time

time_now = time.strftime("%Y%m%d_%H%M%S")
peft_model_path = os.path.join(output_directory, f"peft_model_{time_now}")

trainer.model.save_pretrained(peft_model_path)
print(f"Modèle LoRA sauvegardé dans : {peft_model_path}")

## Étape 8 : Recharger le modèle LoRA pour l'inférence

`is_trainable=False` car on ne veut plus entraîner le modèle, seulement l'utiliser pour générer du texte.

In [ ]:
from peft import PeftModel

loaded_model = PeftModel.from_pretrained(
    foundation_model,
    peft_model_path,
    is_trainable=False
)

## Étape 9 : Générer du texte avec le modèle fine-tuné

In [ ]:
inputs = tokenizer("Two things are infinite: ", return_tensors="pt")

outputs = loaded_model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=50,
    do_sample=True,
    top_p=0.9,
    temperature=0.8,
    pad_token_id=tokenizer.eos_token_id
)

print(tokenizer.batch_decode(outputs, skip_special_tokens=True))

## Pour aller plus loin (critique honnête de l'exercice)

Ne t'attends pas à un résultat impressionnant : avec `r=1`, un échantillon minuscule et 5 epochs, le modèle a à peine eu de quoi apprendre le style des citations. C'est normal, et c'est voulu par l'exercice pour aller vite sur CPU.

Si tu veux un vrai résultat exploitable :
- Augmente `r` à 8 ou 16.
- Baisse `learning_rate` à quelque chose comme `1e-4` ou `2e-4` (3e-2 est très agressif, typique pour LoRA mais souvent trop haut pour ce genre de modèle).
- Utilise tout le dataset (ou au moins 50%), pas 10%.
- Augmente le nombre d'epochs.
- Idéalement, entraîne sur GPU plutôt que CPU (`use_cpu=False` si dispo).